# Lab 7 – HMM-based Noun Phrase Chunking

Tags: **B-NP** (Beginning), **I-NP** (Inside), **O** (Other)

Tasks:
1. Forward Algorithm – probability of each test sentence
2. Viterbi Decoding  – predicted NP tags + evaluation via seqeval

## 1. Load Data

In [1]:
def load_data(filepath):
    sentences = []
    current = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.strip() == '':
                if current:
                    sentences.append(current)
                    current = []
            else:
                parts = line.split('\t')
                word, tag = parts[0], parts[1]
                current.append((word, tag))
    if current:
        sentences.append(current)
    return sentences

train_data = load_data(r'Data/train_NP.txt')
test_data  = load_data(r'Data/test_NP.txt')

print(f'Train sentences : {len(train_data)}')
print(f'Test  sentences : {len(test_data)}')
print('\nFirst training sentence:')
for tok in train_data[0]:
    print(tok)


Train sentences : 823
Test  sentences : 77

First training sentence:
('Rockwell', 'B-NP')
('International', 'I-NP')
('Corp.', 'I-NP')
("'s", 'B-NP')
('Tulsa', 'I-NP')
('unit', 'I-NP')
('said', 'O')
('it', 'B-NP')
('signed', 'O')
('a', 'B-NP')
('tentative', 'I-NP')
('agreement', 'I-NP')
('extending', 'O')
('its', 'B-NP')
('contract', 'I-NP')
('with', 'O')
('Boeing', 'B-NP')
('Co.', 'I-NP')
('to', 'O')
('provide', 'O')
('structural', 'B-NP')
('parts', 'I-NP')
('for', 'O')
('Boeing', 'B-NP')
("'s", 'B-NP')
('747', 'I-NP')
('jetliners', 'I-NP')
('.', 'O')


## 2. Train HMM (Add-K Smoothing, K=1)

In [2]:
from collections import defaultdict
import math

states      = ['B-NP', 'I-NP', 'O']
init_count  = defaultdict(int)
trans_count = defaultdict(lambda: defaultdict(int))
emit_count  = defaultdict(lambda: defaultdict(int))
state_count = defaultdict(int)
vocab       = set()

for sentence in train_data:
    for i, (word, tag) in enumerate(sentence):
        vocab.add(word)
        emit_count[tag][word] += 1
        state_count[tag] += 1
        if i == 0:
            init_count[tag] += 1
        else:
            prev_tag = sentence[i-1][1]
            trans_count[prev_tag][tag] += 1

vocab.add('<UNK>')
V = len(vocab)
K = 1         # Add-K smoothing constant
N = len(train_data)

print(f'States : {states}')
print(f'Vocab  : {V} tokens')
print(f'Tag counts: { {s: state_count[s] for s in states} }')


States : ['B-NP', 'I-NP', 'O']
Vocab  : 4571 tokens
Tag counts: {'B-NP': 5051, 'I-NP': 5727, 'O': 8394}


In [3]:
def log_init_prob(tag):
    return math.log((init_count[tag] + K) / (N + K * len(states)))

def log_trans_prob(prev_tag, cur_tag):
    denom = state_count[prev_tag] + K * len(states)
    return math.log((trans_count[prev_tag][cur_tag] + K) / denom)

def log_emit_prob(tag, word):
    if word not in vocab:
        word = '<UNK>'
    denom = state_count[tag] + K * V
    return math.log((emit_count[tag][word] + K) / denom)

print('log P(B-NP | START) :', round(log_init_prob('B-NP'), 4))
print('log P(I-NP | B-NP)  :', round(log_trans_prob('B-NP', 'I-NP'), 4))
print('log P("the" | B-NP) :', round(log_emit_prob('B-NP', 'the'), 4))


log P(B-NP | START) : -0.4213
log P(I-NP | B-NP)  : -0.3634
log P("the" | B-NP) : -2.4772


## 3. Task 1 – Forward Algorithm

alpha_t(j) = P(o_1,...,o_t, q_t=s_j | lambda)  
P(O | lambda) = sum_j alpha_T(j)

In [4]:
def forward(sentence):
    """
    Compute log P(sentence) using the Forward Algorithm (log-space).
    Returns log_prob and the alpha table.
    """
    T = len(sentence)
    alpha = [None] * T

    # Initialisation
    alpha[0] = {s: log_init_prob(s) + log_emit_prob(s, sentence[0]) for s in states}

    # Recursion
    for t in range(1, T):
        alpha[t] = {}
        for s in states:
            log_vals = [alpha[t-1][prev] + log_trans_prob(prev, s) for prev in states]
            max_val  = max(log_vals)
            log_sum  = max_val + math.log(sum(math.exp(v - max_val) for v in log_vals))
            alpha[t][s] = log_sum + log_emit_prob(s, sentence[t])

    # Termination
    final_vals = [alpha[T-1][s] for s in states]
    max_val    = max(final_vals)
    log_prob   = max_val + math.log(sum(math.exp(v - max_val) for v in final_vals))

    return log_prob, alpha

print('Forward algorithm defined.')


Forward algorithm defined.


In [5]:
print(f'{"Sent":>6}  Log P(sentence)')
print('-' * 35)

log_probs = []
for idx, sentence_pairs in enumerate(test_data):
    words = [w for w, _ in sentence_pairs]
    lp, _ = forward(words)
    log_probs.append(lp)
    print(f'{idx+1:>6}   {lp:.6f}')

print()
print(f'Average log-probability : {sum(log_probs)/len(log_probs):.6f}')


  Sent  Log P(sentence)
-----------------------------------
     1   -263.677133
     2   -188.571679
     3   -213.488657
     4   -243.887357
     5   -218.915455
     6   -253.472159
     7   -273.391495
     8   -178.810092
     9   -202.988010
    10   -170.706751
    11   -115.176814
    12   -169.500711
    13   -184.355635
    14   -336.630602
    15   -164.172759
    16   -79.098495
    17   -218.928666
    18   -85.175081
    19   -136.784754
    20   -159.038336
    21   -156.971245
    22   -258.679569
    23   -263.099726
    24   -188.804265
    25   -189.994657
    26   -146.407589
    27   -124.581204
    28   -143.809001
    29   -284.501028
    30   -33.421513
    31   -93.900436
    32   -285.353952
    33   -262.702515
    34   -174.398572
    35   -147.531847
    36   -170.629652
    37   -159.419331
    38   -64.121588
    39   -121.429787
    40   -148.921296
    41   -219.598466
    42   -163.296548
    43   -75.033607
    44   -83.585321
    45   -62.197590
   

## 4. Task 2 – Viterbi Decoding

In [6]:
def viterbi(sentence):
    """
    Viterbi decoding – returns the most-likely tag sequence.
    """
    T = len(sentence)
    vit  = [None] * T
    back = [None] * T

    # Initialisation
    vit[0]  = {s: log_init_prob(s) + log_emit_prob(s, sentence[0]) for s in states}
    back[0] = {s: None for s in states}

    # Recursion
    for t in range(1, T):
        vit[t]  = {}
        back[t] = {}
        for s in states:
            scores    = {prev: vit[t-1][prev] + log_trans_prob(prev, s) for prev in states}
            best_prev = max(scores, key=scores.get)
            vit[t][s]  = scores[best_prev] + log_emit_prob(s, sentence[t])
            back[t][s] = best_prev

    # Termination
    best_last = max(states, key=lambda s: vit[T-1][s])

    # Backtrack
    path = [best_last]
    for t in range(T-1, 0, -1):
        path.append(back[t][path[-1]])
    path.reverse()
    return path

print('Viterbi algorithm defined.')


Viterbi algorithm defined.


In [7]:
true_tags = []
pred_tags = []

for sentence_pairs in test_data:
    words = [w for w, _ in sentence_pairs]
    gold  = [t for _, t in sentence_pairs]
    pred  = viterbi(words)
    true_tags.append(gold)
    pred_tags.append(pred)

print('Viterbi predictions computed.')
print('\nExample – first test sentence:')
words_ex = [w for w, _ in test_data[0]]
print(f'{"Word":<20} {"Gold":<8} {"Pred"}')
print('-' * 38)
for w, g, p in zip(words_ex, true_tags[0], pred_tags[0]):
    print(f'{w:<20} {g:<8} {p}')


Viterbi predictions computed.

Example – first test sentence:
Word                 Gold     Pred
--------------------------------------
Confidence           B-NP     B-NP
in                   O        O
the                  B-NP     B-NP
pound                I-NP     I-NP
is                   O        O
widely               O        B-NP
expected             O        O
to                   O        O
take                 O        O
another              B-NP     B-NP
sharp                I-NP     I-NP
dive                 I-NP     I-NP
if                   O        O
trade                B-NP     B-NP
figures              I-NP     I-NP
for                  O        O
September            B-NP     B-NP
,                    O        O
due                  O        O
for                  O        O
release              B-NP     B-NP
tomorrow             B-NP     I-NP
,                    O        O
fail                 O        B-NP
to                   O        O
show                 O   

## 5. Evaluation with seqeval

In [8]:
try:
    import seqeval
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seqeval', '-q'])

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print(classification_report(true_tags, pred_tags))
print(f'Overall F1        : {f1_score(true_tags, pred_tags):.4f}')
print(f'Overall Precision : {precision_score(true_tags, pred_tags):.4f}')
print(f'Overall Recall    : {recall_score(true_tags, pred_tags):.4f}')


              precision    recall  f1-score   support

          NP       0.64      0.73      0.68       475

   micro avg       0.64      0.73      0.68       475
   macro avg       0.64      0.73      0.68       475
weighted avg       0.64      0.73      0.68       475

Overall F1        : 0.6818
Overall Precision : 0.6425
Overall Recall    : 0.7263
